# 02 — Retrieval Models

This notebook demonstrates the three CPU-only retrieval models in the repo:

| Model | Description |
|-------|-------------|
| `PopularityModel` | Global ranking by distinct purchasing customers |
| `ItemToItem` | Co-purchase cosine similarity |
| `ALSModel` | Alternating Least Squares (via `implicit`) |

It also shows how to build a FAISS index over ALS item factors for approximate
nearest-neighbour lookup.

All models run on the committed sample — no GPU or download required.


In [1]:
"""Copy-paste this as the first cell of every notebook in a portfolio project.

It walks up from the notebook's current working directory to find the project
root (the folder containing requirements.txt + src/), chdirs there, and adds
it to sys.path. After this cell runs, the notebook can:

  - import from src.x.y      (would otherwise fail with ModuleNotFoundError)
  - read relative paths like 'data/processed/...' or 'docs/...' regardless of
    where the notebook was launched from (notebooks/, project root, anywhere)
  - work the same whether launched from Jupyter, VS Code, jupyter nbconvert,
    or Colab (assuming the project was extracted to a known location there)
"""

# fmt: off
import os, sys
from pathlib import Path

_cwd = Path.cwd()
_root = next(
    (p for p in [_cwd] + list(_cwd.parents)
     if (p / 'requirements.txt').exists() and (p / 'src').is_dir()),
    None,
)
assert _root, f'Could not find project root from {_cwd}. Open the notebook from inside the project tree.'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')
# fmt: on


Project root: C:\Users\markd\OneDrive\Desktop\Claude\GitHub\Projects\uk-retail-recommender


In [2]:
import pandas as pd

from src.data.interactions import build_interactions
from src.data.split import temporal_split
from src.retrieval.popularity import PopularityModel
from src.retrieval.item2item import ItemToItem
from src.retrieval.als import ALSModel
from src.retrieval.faiss_index import FaissIndex

# Load committed sample and do a temporal split
df = pd.read_parquet("tests/fixtures/sample_transactions.parquet")
cutoff = df["date"].median()
train_df, test_df = temporal_split(df, cutoff)
print(f"Train rows : {len(train_df):,}  ({train_df['date'].min().date()} – {train_df['date'].max().date()})")
print(f"Test rows  : {len(test_df):,}  ({test_df['date'].min().date()} – {test_df['date'].max().date()})")


Train rows : 1,175  (2010-01-02 – 2010-11-07)
Test rows  : 1,116  (2010-11-08 – 2011-08-23)


## Popularity model — global top-10

In [3]:
pop = PopularityModel().fit(train_df)
top10 = pop.recommend(k=10)

pop_df = pd.DataFrame(top10, columns=["item_id", "distinct_customers"])
print("Top 10 most popular items:")
print(pop_df.to_string(index=False))


Top 10 most popular items:
item_id  distinct_customers
 SKU016                21.0
 SKU003                18.0
 SKU075                17.0
 SKU076                17.0
 SKU036                16.0
 SKU052                16.0
 SKU051                16.0
 SKU006                15.0
 SKU022                15.0
 SKU028                15.0


## Item-to-item — similar items for an example seed

In [4]:
i2i = ItemToItem().fit(train_df)

# Pick the most popular item as a seed
seed_item = pop.recommend(k=1)[0][0]
neighbours = i2i.similar(seed_item, k=10)

print(f"Item-to-item neighbours for '{seed_item}':")
i2i_df = pd.DataFrame(neighbours, columns=["neighbour", "similarity"])
print(i2i_df.to_string(index=False))


Item-to-item neighbours for 'SKU016':
neighbour  similarity
   SKU010    0.279946
   SKU064    0.249136
   SKU046    0.170406
   SKU040    0.167968
   SKU022    0.166091
   SKU058    0.143839
   SKU004    0.135113
   SKU028    0.131306
   SKU034    0.123797
   SKU014    0.099258


## ALS — recommendations for an example customer

In [5]:
inter = build_interactions(train_df)
als = ALSModel(factors=32, iterations=8).fit(inter)

# Pick an example customer from the interaction matrix
example_user = inter.user_ids[0]
als_recs = als.recommend(example_user, k=10)

print(f"ALS recommendations for customer {example_user}:")
als_df = pd.DataFrame(als_recs, columns=["item_id", "score"])
print(als_df.to_string(index=False))


ALS recommendations for customer 1000:
item_id    score
 SKU017 0.482339
 SKU011 0.376625
 SKU018 0.322800
 SKU001 0.307102
 SKU039 0.275525
 SKU045 0.246415
 SKU046 0.236693
 SKU041 0.216927
 SKU070 0.216127
 SKU029 0.202581


C:\Users\markd\OneDrive\Desktop\Claude\GitHub\Projects\uk-retail-recommender\.venv\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


## FAISS — approximate nearest neighbour over ALS item factors

Build a FAISS `IndexFlatIP` over the L2-normalised ALS item factor matrix
and query for items similar to an example seed item.


In [6]:
item_factors = als.item_factors
faiss_idx = FaissIndex(dim=item_factors.shape[1]).build(item_factors, inter.item_ids)

# Query with the factor vector for the seed item
seed_idx = inter.item_index[seed_item]
query_vec = item_factors[seed_idx]
faiss_results = faiss_idx.query(query_vec, k=10)

print(f"FAISS nearest neighbours for item '{seed_item}' (ALS factor space):")
faiss_df = pd.DataFrame(faiss_results, columns=["item_id", "cosine_score"])
print(faiss_df.to_string(index=False))


FAISS nearest neighbours for item 'SKU016' (ALS factor space):
item_id  cosine_score
 SKU016      1.000000
 SKU010      0.379378
 SKU019      0.319951
 SKU071      0.260385
 SKU064      0.238563
 SKU070      0.235177
 SKU052      0.207195
 SKU049      0.182112
 SKU063      0.181200
 SKU042      0.149547


## Commentary

- **Popularity** is a strong zero-personalisation baseline; it requires no
  customer history.
- **Item-to-item** exploits co-purchase patterns and improves for customers
  with at least a few basket items to aggregate over.
- **ALS** learns latent factors and provides personalised scores. At this
  sample scale (120 customers, 80 items) the improvement over i2i is modest;
  on the full dataset it is more pronounced.
- **FAISS** wraps ALS factors into an approximate-nearest-neighbour index.
  On the full item catalogue (thousands of SKUs) this is the practical
  serving path — exact linear scan becomes too slow.
